# Lezione 7 — Grafici interattivi con Plotly

Finora i tuoi grafici erano immagini statiche: li generi, li guardi, e restano lì. Se vuoi sapere il valore esatto di un punto, devi indovinarlo guardando l'asse. Se il grafico ha molti dati ammassati, non puoi avvicinarti per vederli meglio.

Nella vita reale — e in qualsiasi report o dashboard che consegni a un'altra persona — questo non basta. Chi guarda il grafico vuole passare il mouse su un punto e vedere il suo valore esatto, fare zoom su una zona specifica, o nascondere una serie con un clic. Questo è esattamente ciò che ti dà **Plotly**.

**Analogia:** passare da matplotlib/seaborn a Plotly è come passare da una foto stampata a una foto sul tuo cellulare. La foto stampata la guardi e basta. La foto sul cellulare puoi toccarla, fare zoom con le dita, scorrerla. Il contenuto può essere lo stesso, ma l'esperienza di esplorarlo cambia completamente.

## Due modi di usare Plotly

Plotly in Python ha due facce, ed è bene sapere quale usare e quando:

- **`plotly.express`** (si importa come `px`): interfaccia di alto livello. Con una sola riga di codice crei un grafico completo, titolo e assi inclusi. È il punto di partenza consigliato quando inizi: meno codice, meno attrito.
- **`plotly.graph_objects`** (si importa come `go`): interfaccia di basso livello. Costruisci il grafico pezzo per pezzo — prima una figura vuota, poi ci aggiungi "tracce" (`traces`), poi regoli il layout (`layout`). Ti dà più controllo, ma richiede più righe.

Internamente, `plotly.express` usa in realtà `plotly.graph_objects`: è un involucro (wrapper) che ti risparmia lavoro per i casi più comuni.

Vedrai entrambe in questo capitolo, usando il dataset delle compagnie aeree (`airline_data.csv`, con informazioni sui voli domestici negli USA) che già conosci dai capitoli precedenti.

`plotly` viene preinstallato in Google Colab, quindi non serve installarlo con `pip install`.

## 0. Preparazione: librerie e dati

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Carichiamo il dataset delle compagnie aeree
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str}
)

print('Dati caricati correttamente:', airline_data.shape)
airline_data.head()

In [ ]:
# Il dataset completo è enorme (centinaia di migliaia di righe), quindi prendiamo
# un campione casuale di 500 righe per lavorare più velocemente.
# random_state=42 assicura di ottenere sempre lo stesso campione.
data = airline_data.sample(n=500, random_state=42)

data.shape

## 1. Scatter plot con `graph_objects`

Uno scatter (dispersione) mostra la relazione tra due variabili. Qui usiamo l'interfaccia di basso livello per vedere come si costruisce un grafico passo per passo: figura vuota → `add_trace()` → `update_layout()`.

In [ ]:
# Passo 1: creiamo una figura vuota
fig = go.Figure()

# Passo 2: aggiungiamo una "traccia" di tipo scatter con add_trace().
# mode='markers' disegna punti (invece di linee).
fig.add_trace(go.Scatter(
    x=data['Distance'],
    y=data['DepTime'],
    mode='markers',
    marker=dict(color='red')
))

# Passo 3: aggiorniamo il titolo e le etichette degli assi
fig.update_layout(
    title='Distance vs Departure Time',
    xaxis_title='Distance',
    yaxis_title='DepTime'
)

fig.show()

Con `fig.show()` ottieni un grafico dove puoi passare il mouse su ogni punto e vedere i suoi valori esatti, o fare zoom trascinando il mouse su una zona.

**Nota:** un line plot si costruisce con lo stesso `go.Scatter()`, cambiando solo `mode='markers'` con `mode='lines'`.

## 2. Bar chart con `plotly.express`

Passiamo all'interfaccia di alto livello. Con `px`, definisci tutto — dati, assi e titolo — in un'unica chiamata.

In [ ]:
# Raggruppiamo per stato di destinazione e sommiamo il totale dei voli
bar_data = data.groupby(['DestState'])['Flights'].sum().reset_index()

# Una sola riga: passiamo il dataframe, x, y e il titolo
fig = px.bar(bar_data, x="DestState", y="Flights",
             title='Total number of flights to the destination state split by reporting airline')
fig.show()

Confronta questo con lo scatter di prima: stesso risultato interattivo, molto meno codice.

## 3. Istogramma con `plotly.express`

Un istogramma raggruppa dati continui in intervalli (bin) e mostra quanti valori cadono in ogni intervallo. Utile per vedere la forma di una distribuzione.

In [ ]:
# Sostituiamo i valori mancanti con 0 prima di creare il grafico
data['ArrDelay'] = data['ArrDelay'].fillna(0)

fig = px.histogram(data, x="ArrDelay",
                    title="Total number of flights to the destination state split by reporting air.")
fig.show()

## 4. Bubble chart: lo scatter con una terza dimensione

Un bubble chart è uno scatter plot a cui aggiungi una terza variabile: la dimensione di ogni punto (la "bolla"). Così, in un unico grafico comunichi tre cose contemporaneamente invece di due.

Nota che un bubble chart in Plotly Express **non ha una propria funzione**: si crea con `px.scatter()` e il parametro `size`.

In [ ]:
# Raggruppiamo per compagnia aerea e sommiamo il numero di voli
bub_data = data.groupby('Reporting_Airline')['Flights'].sum().reset_index()

# size="Flights" fa sì che la dimensione di ogni bolla rappresenti il numero di voli.
# hover_name definisce quale testo appare in grassetto al passaggio del mouse.
fig = px.scatter(bub_data, x="Reporting_Airline", y="Flights", size="Flights",
                  hover_name="Reporting_Airline",
                  title='Reporting Airline vs Number of Flights', size_max=60)
fig.show()

## 5. Pie chart

Un pie chart (grafico a torta) mostra quale proporzione rappresenta ogni categoria all'interno di un totale.

In [ ]:
# values definisce la dimensione di ogni fetta, names definisce la sua etichetta
fig = px.pie(data, values='Month', names='DistanceGroup',
             title='Distance group proportion by month')
fig.show()

## 6. Sunburst: quando una gerarchia non entra in un pie chart

Tutti i grafici precedenti confrontano categorie a un solo livello. Ma cosa succede se i tuoi dati hanno livelli annidati, come mese → stato di destinazione → numero di voli? È lì che il pie chart non basta più ed entra in gioco il **sunburst**.

Un sunburst rappresenta dati gerarchici con cerchi concentrici: il cerchio più interno è la radice (il livello più generale), e ogni anello verso l'esterno rappresenta un livello più specifico della gerarchia. È come un pie chart, ma con "strati" verso l'esterno che puoi esplorare cliccando per fare zoom su un ramo specifico.

In [ ]:
# path definisce l'ordine della gerarchia: prima Month (cerchio interno),
# poi DestStateName (anello esterno). values definisce la dimensione di ogni settore.
fig = px.sunburst(data, path=['Month', 'DestStateName'], values='Flights',
                   title='Flight Distribution Hierarchy')
fig.show()

Passando il mouse sul cerchio interno vedi il totale dei voli di quel mese; guardando l'anello esterno vedi come si distribuisce quel totale tra i diversi stati di destinazione. Questo — mostrare un totale e la sua ripartizione in un unico grafico interattivo — è ciò che un bar chart o un pie chart normali non ti danno in modo diretto.

## Punti chiave

- Plotly trasforma grafici statici in grafici interattivi: puoi fare zoom, passare il mouse per vedere valori esatti, ed esplorare i dati invece di limitarti a guardarli.
- `plotly.express` (`px`) è l'interfaccia di alto livello: meno codice, buon punto di partenza.
- `plotly.graph_objects` (`go`) è l'interfaccia di basso livello: figura vuota → `add_trace()` → `update_layout()`. Più controllo, più righe.
- Scatter e line plot in `graph_objects` usano la stessa funzione (`go.Scatter`); cambia solo il parametro `mode` (`'markers'` vs `'lines'`).
- Un bubble chart è uno scatter con una terza variabile codificata nella dimensione (`size=`).
- Un sunburst mostra gerarchie (più livelli annidati) con cerchi concentrici — qualcosa che un pie chart a un solo livello non può rappresentare.

## Prossimo passo

Apri `clase07_practica_plotly-interactivo_IT.ipynb` e risolvi gli esercizi proposti.